# Fine-tuning a Nemotron 3 text embedding model


In [ ]:
!pip install -q -U bitsandbytes "torchao>=0.16.0"


In [ ]:
from datasets import load_dataset

print("Loading dataset...")
raw_ds = load_dataset("nvidia/Retrieval-Synthetic-NVDocs-v1",
                      split="train[:5000]")

sample_ds = raw_ds.shuffle(seed=42).select(range(min(3000, len(raw_ds))))

print(f"Sampled {len(sample_ds)} random examples.")


### Domain-Specific Retrieval Datasets Overview

| Dataset | Why it fits | Hugging Face URL |
| :--- | :--- | :--- |
| nvidia/Retrieval-Synthetic-NVDocs-v1 | Synthetic retrieval over NVIDIA documents. | https://huggingface.co/datasets/nvidia/Retrieval-Synthetic-NVDocs-v1 |
| Taylor658/synthetic-legal | Synthetic legal text dataset | https://huggingface.co/datasets/Taylor658/synthetic-legal |
| microsoft/mediflow | Large medical instruction dataset with retrieval/IR-adjacent examples and clinically grounded text. | https://huggingface.co/datasets/microsoft/mediflow |
| abhinand/MedEmbed-training-triplets-v1 | Synthetic triplets for medical retrieval training | https://huggingface.co/datasets/abhinand/MedEmbed-training-triplets-v1 |
| HFforLegal/case-law | Large legal corpus useful for retrieval indexing and RAG over legal decisions. | https://huggingface.co/datasets/HFforLegal/case-law |
| isaacus/legal-rag-bench | Retrieval-focused legal RAG dataset from a legal collection. | https://huggingface.co/datasets/isaacus/legal-rag-bench |
| isaacus/legal-rag-qa | Legal RAG QA dataset, good for retrieval-style benchmarking. | https://huggingface.co/datasets/isaacus/legal-rag-qa |
| isaacus/contractual-clause-retrieval | Explicit legal retrieval dataset centered on clause search. | https://huggingface.co/datasets/isaacus/contractual-clause-retrieval |
| PatronusAI/financebench | Finance-domain benchmark often used for retrieval over financial documents. | https://huggingface.co/datasets/PatronusAI/financebench |
| c3po-ai/edgar-corpus | SEC filings corpus, useful for long-document financial retrieval. | https://huggingface.co/datasets/c3po-ai/edgar-corpus |


In [ ]:
import torch
import warnings
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
    GenerationConfig
)
import random
import numpy as np

warnings.filterwarnings("ignore")

qwen_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
llm_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {llm_id}...")
tokenizer = AutoTokenizer.from_pretrained(llm_id, clean_up_tokenization_spaces=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id, quantization_config=qwen_bnb, device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
)

pipe.generation_config = GenerationConfig(
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)


In [ ]:
def generate_synthetic_pair(example, chunk_length=1500):

    doc = str(example.get("text", "")).strip()
    if len(doc) > chunk_length:
        random_start = random.randint(0, len(doc) - chunk_length)
        start_pos = doc.find(". ", random_start)
        start_idx = start_pos + 2 if start_pos != -1 else random_start
        end_pos = doc.find(".", start_idx + chunk_length)
        end_idx = end_pos + 1 if end_pos != -1 else len(doc)
        chunk = doc[start_idx:end_idx].strip()
    else:
        chunk = doc

    prompt = f"<|im_start|>user\nRead the following document chunk and generate a single, short, specific question that is directly answered by it.\n\nDocument: {chunk}\n\nQuestion:<|im_end|>\n<|im_start|>assistant\n"

    outputs = pipe(
        prompt,
        truncation=True,
        return_full_text=False,
    )
    question = outputs[0]["generated_text"].strip()

    return {"anchor": question, "positive": chunk}


In [ ]:
print("Generating synthetic pairs (this may take a moment)...")
pair_ds = sample_ds.map(generate_synthetic_pair, remove_columns=sample_ds.column_names)
print("Generation Complete! Here are 3 examples:")
for i in range(min(3, len(pair_ds))):
    print(f"\n--- Example {i + 1} ---")
    print("Q:", pair_ds[i]["anchor"])
    print("A:", pair_ds[i]["positive"][:200], "...")

split_ds = pair_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]


In [ ]:
from transformers import AutoConfig
from sentence_transformers import SentenceTransformer
from peft import get_peft_model, LoraConfig, TaskType

embed_id = "nvidia/Nemotron-3-Embed-1B-BF16"

print(f"Loading base embedding model {embed_id}...")
embed_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16}, 
    )

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,)

peft_model = get_peft_model(embed_model[0].auto_model, lora_config)
embed_model[0].auto_model = peft_model
print("\nLoRA trainable parameters:")
peft_model.print_trainable_parameters()


In [ ]:
from sentence_transformers.sentence_transformer.losses import (
    MultipleNegativesRankingLoss,)
from sentence_transformers.sentence_transformer.training_args import (
    SentenceTransformerTrainingArguments,)
from sentence_transformers.sentence_transformer.trainer import (
    SentenceTransformerTrainer,)

loss = MultipleNegativesRankingLoss(embed_model)

args = SentenceTransformerTrainingArguments(
    output_dir="./nemotron-embed-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",)

trainer = SentenceTransformerTrainer(
    model=embed_model, args=args, train_dataset=train_ds, loss=loss)

trainer.train()


In [ ]:
eval_queries = eval_ds["anchor"]
eval_docs = eval_ds["positive"]

def evaluate_retrieval(model, model_name):
    query_embs = model.encode(eval_queries)
    doc_embs = model.encode(eval_docs)
    similarities = np.dot(query_embs, doc_embs.T)
    top_doc_indices = np.argmax(similarities, axis=1)
    correct_indices = np.arange(len(eval_queries))
    correct_retrievals = np.sum(top_doc_indices == correct_indices)
    recall_at_1 = correct_retrievals / len(eval_queries)
    print(f"{model_name} Strict Recall@1: {recall_at_1 * 100:.2f}%")
    return recall_at_1

print("\n--- Evaluation Results ---")

original_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16})
finetuned_model = embed_model

original_score = evaluate_retrieval(original_model, "Original Model")
finetuned_score = evaluate_retrieval(finetuned_model, "Fine-Tuned Model")

improvement = finetuned_score - original_score
print(f"Absolute Improvement: +{improvement * 100:.2f}%")
